In [8]:
import numpy as np
import cvxpy as cp
from scipy import optimize

In [9]:
means = np.array([0.05, 0.06, 0.08, 0.06])
vols = np.array([0.15, 0.2, 0.25, 0.3])
vol_mat = np.diag(vols)
correlation_mat = np.array(
    [
        [1.0, 0.1, 0.4, 0.5],
        [0.1, 1.0, 0.7, 0.4],
        [0.4, 0.7, 1.0, 0.8],
        [0.5, 0.4, 0.8, 1.0],
    ]
)
cov_matrix = vol_mat @ correlation_mat @ vol_mat
cov_matrix

array([[0.0225, 0.003 , 0.015 , 0.0225],
       [0.003 , 0.04  , 0.035 , 0.024 ],
       [0.015 , 0.035 , 0.0625, 0.06  ],
       [0.0225, 0.024 , 0.06  , 0.09  ]])

In [10]:
sr = 0.25
weight = np.array([0.4, 0.3, 0.2, 0.1])
vol = np.sqrt(w @ cov_matrix @ weight)
phi = sr / vol
rf = 0.03
phi

1.6287422626782593

In [59]:
mu_est = rf + sr * (cov_matrix @ weight) / vol
mu_est

array([0.05467545, 0.06680958, 0.08700598, 0.09058921])

In [124]:
p = np.array([[1, 0, 0, 0], [0, 1, -1, 0]])
q = np.array([0.04, -0.01])
# p = np.array([[0, 0, 0, 0], [0, 0, 0, 0]])
# q = np.array([0.0, 0.0])
omega = np.array([[0.1, 0], [0, 0.05]]) ** 2
# omega = np.array([[0.0, 0], [0, 0.0]]) ** 2
tau = 1.0
big_gamma = tau * cov_matrix

mu_given_view = mu_est + big_gamma @ p.T @ np.linalg.inv(
    p @ big_gamma @ p.T + omega
) @ (q - p @ mu_est)
mu_given_view

array([0.04393844, 0.06640951, 0.07680025, 0.07610116])

In [125]:
w = cp.Variable(len(mu_given_view))
b = np.array([0.4, 0.3, 0.2, 0.1])
gamma = cp.Parameter(nonneg=True)
gamma.value = 1 / phi
ret = mu_given_view.T @ (w - b)
risk = cp.quad_form((w - b), cov_matrix)
prob = cp.Problem(cp.Minimize(risk - gamma * ret), [cp.sum(w) == 1, w >= 0])
prob.solve()
optimal_vol = cp.sqrt(risk).value
optimal_ret = ret.value
print("Optimal return:", optimal_ret, "Optimal vol:", optimal_vol)
print("Optimal weights:", w.value)

Optimal return: 0.006314542767438159 Optimal vol: 0.044028082730790064
Optimal weights: [0.20333221 0.31046277 0.32955884 0.15664618]
